---
title: 'Lab 9: Autograd, `nn.Module` i pierwsze sieci neuronowe'
subtitle: Biblioteki Python w analizie danych
author: Tomasz Rodak
jupyter: python3
---


[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_9.ipynb)

Na lab 8 trenowaliśmy regresję liniową i logistyczną w PyTorch, ale gradient funkcji straty wyprowadzaliśmy ręcznie i programowali jako wyrażenie macierzowe. Wykład 5 pokazał, że to już niepotrzebne: mechanizm różniczkowania automatycznego (*autograd*) buduje graf obliczeniowy w trakcie forward passu i sam liczy wszystkie gradienty po wywołaniu `loss.backward()`.

W tym arkuszu wykorzystamy autograd w trzech narastających kontekstach. Najpierw na małej funkcji matematycznej — przejdziemy backpropagację ręcznie, węzeł po węźle, i porównamy z wynikiem PyTorcha. Potem przepiszemy regresję logistyczną z lab 8 tak, by gradient liczył się sam. Następnie zrefaktoryzujemy ten sam model używając `nn.Module` i `optim` — wprowadzając kanoniczną pętlę treningową, którą będziemy już zawsze wykorzystywać. Wreszcie wytrenujemy pierwsze sieci neuronowe z prawdziwego zdarzenia: MLP na regresji (sunspoty z lab 8) i MLP na klasyfikacji wieloklasowej (zbiór 2D z czterema klasami), na którym zobaczymy, jak warstwa ukryta zmienia kształt granic decyzyjnych.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

## 1. Rozgrzewka: graf obliczeniowy i autograd

Na wykładzie 5 widzieliśmy, że każda funkcja zapisana jako sekwencja operacji elementarnych daje się przedstawić jako skierowany graf acykliczny i że algorytm wstecznej propagacji to systematyczne stosowanie reguły łańcuchowej w odwrotnej kolejności topologicznej tego grafu. W tej sekcji przejdziemy ten algorytm ręcznie na małym przykładzie, a następnie zweryfikujemy wynik autograd-em.

### 1.1 Funkcja i jej graf obliczeniowy

Rozważmy funkcję trzech zmiennych:

$$L(x_1, x_2, x_3) = (x_1 + x_2) \sin(x_2 x_3).$$

Rozłóżmy ją na operacje elementarne, wprowadzając węzły pośrednie $a$, $b$, $c$:

$$
a = x_1 + x_2, \qquad
b = x_2 \, x_3, \qquad
c = \sin(b), \qquad
L = a \cdot c.
$$

Korzeniem grafu jest $L$, liśćmi — zmienne $x_1, x_2, x_3$. Krawędzie kierowane "w dół", od korzenia do liści, opisują **relację rodzic → dziecko** w grafie obliczeniowym: jeśli wartość węzła $u$ jest obliczana na podstawie wartości węzła $v$, to $u$ jest **rodzicem** $v$, a $v$ jego **dzieckiem**. (Forward pass biegnie w odwrotnym kierunku — od dzieci do rodziców.)

Krawędzie naszego grafu:

| dziecko | rodzic | operacja przy rodzicu |
|---------|--------|------------------------|
| $a$     | $L$    | $L = a \cdot c$        |
| $c$     | $L$    | $L = a \cdot c$        |
| $b$     | $c$    | $c = \sin(b)$          |
| $x_1$   | $a$    | $a = x_1 + x_2$        |
| $x_2$   | $a$    | $a = x_1 + x_2$        |
| $x_2$   | $b$    | $b = x_2 \, x_3$       |
| $x_3$   | $b$    | $b = x_2 \, x_3$       |

Zauważ, że **$x_2$ ma dwóch rodziców** ($a$ oraz $b$) — pojawia się w dwóch ścieżkach prowadzących od liścia do korzenia. To kluczowy szczegół, do którego za chwilę wrócimy.

### 1.2 Forward pass z konkretnymi wartościami

Ustalmy punkt $(x_1, x_2, x_3) = (1, 2, 3)$ i policzmy wartości wszystkich węzłów:

1. $a = 1 + 2 = 3$.
2. $b = 2 \cdot 3 = 6$.
3. $c = \sin(6) \approx -0.2794$.
4. $L = a \cdot c = 3 \sin(6) \approx -0.8382$.

Te liczby przydadzą się w 1.5 do weryfikacji numerycznej.

### 1.3 Backward pass — definicje i reguła

Dla każdego węzła $v$ w grafie wprowadzamy oznaczenie

$$\bar v \;:=\; \frac{\partial L}{\partial v}.$$

Reguła wstecznej propagacji mówi, że dla każdego węzła $v$ różnego od korzenia

$$\bar v \;=\; \sum_{u \in \text{rodzice}(v)} \bar u \cdot \frac{\partial u}{\partial v},$$

gdzie suma przebiega po wszystkich rodzicach $v$ w grafie, a pochodne $\partial u / \partial v$ to **lokalne pochodne** — zależne tylko od operacji wykonywanej przy węźle $u$. Dla korzenia kładziemy $\bar L = \partial L / \partial L = 1$.

Algorytm: idziemy w **odwrotnej kolejności topologicznej**, od korzenia do liści. W każdym węźle używamy już policzonych $\bar u$ rodziców i mnożymy przez lokalną pochodną.

### 1.4 Backward pass — ręczne wyprowadzenie

Wykonaj algorytm krok po kroku. Dla każdego węzła zapisz najpierw wzór symboliczny (w którym wykorzystujesz $\bar u$ rodziców i pochodne lokalne), a następnie podstaw wartości $a$, $b$, $c$ z 1.2.

1. $\bar L = 1$.
2. $\bar a$: jedyny rodzic $a$ to $L = a \cdot c$, czyli $\partial L / \partial a = c$. Stąd
   $$\bar a = \bar L \cdot c = c.$$
3. $\bar c$: jedyny rodzic $c$ to $L = a \cdot c$, czyli $\partial L / \partial c = a$. Stąd
   $$\bar c = \bar L \cdot a = a.$$
4. $\bar b$: jedyny rodzic $b$ to $c = \sin(b)$, czyli $\partial c / \partial b = \cos(b)$. Stąd
   $$\bar b = \bar c \cdot \cos(b) = a \cos(b).$$
5. $\bar{x_1}$: jedyny rodzic to $a = x_1 + x_2$, $\partial a / \partial x_1 = 1$. Stąd
   $$\bar{x_1} = \bar a \cdot 1 = c.$$
6. $\bar{x_3}$: jedyny rodzic to $b = x_2 x_3$, $\partial b / \partial x_3 = x_2$. Stąd
   $$\bar{x_3} = \bar b \cdot x_2 = a \, x_2 \cos(b).$$
7. $\bar{x_2}$ — przypadek z dwoma rodzicami. Sumujemy wkład z gałęzi przez $a$ i z gałęzi przez $b$:
   - przez $a$: $\partial a / \partial x_2 = 1$, wkład $\bar a \cdot 1 = c$,
   - przez $b$: $\partial b / \partial x_2 = x_3$, wkład $\bar b \cdot x_3 = a \, x_3 \cos(b)$.

   Stąd
   $$\bar{x_2} = c + a \, x_3 \cos(b).$$

Sumowanie po węzłach nadrzędnych nie jest modyfikacją reguły łańcuchowej, lecz jej naturalną konsekwencją w przypadku funkcji wielu zmiennych. Jeśli zmienna $x$ ma wielu odbiorców, jej łączny gradient jest sumą wpływów przekazywanych przez każdą z tych ścieżek. Backpropagacja po prostu porządkuje to sumowanie, zapewniając efektywność obliczeniową.

Podstawiając wartości z 1.2 ($a = 3$, $b = 6$):

$$
\bar{x_1} = \sin(6), \quad
\bar{x_2} = \sin(6) + 9\cos(6), \quad
\bar{x_3} = 6\cos(6).
$$

Oblicz te wartości liczbowo (np. przez `math.sin`, `math.cos`).

### 1.5 Weryfikacja symboliczna z bezpośredniego różniczkowania

Bez korzystania z grafu, policz $\partial L / \partial x_j$ dla $j = 1, 2, 3$ bezpośrednio z definicji $L = (x_1 + x_2)\sin(x_2 x_3)$ (reguła iloczynu i reguła łańcuchowa). Sprawdź, że dostajesz dokładnie te same wyrażenia, co w 1.4. To wewnętrzny kontrol algorytmu — backpropagacja musi dać identyczny wynik z tym, co dostalibyśmy "klasycznie".

### 1.6 Weryfikacja numeryczna w PyTorch

Zaimplementuj forward pass jako sekwencję operacji tensorowych z `requires_grad=True` na liściach, wywołaj `backward()` i porównaj `.grad` liści z liczbami z 1.4.

In [ ]:
#| eval: false
x1 = torch.tensor(1.0, requires_grad=True)
x2 = torch.tensor(2.0, requires_grad=True)
x3 = torch.tensor(3.0, requires_grad=True)

a = x1 + x2
b = x2 * x3
c = torch.sin(b)
L = a * c

L.backward()

print(x1.grad, x2.grad, x3.grad)

Sprawdź, że wartości zgadzają się z ręcznymi do precyzji float32 — użyj `torch.allclose` z tolerancją rzędu $10^{-6}$, np.:

In [ ]:
#| eval: false
import math
expected = torch.tensor([
    math.sin(6),
    math.sin(6) + 9 * math.cos(6),
    6 * math.cos(6),
])
actual = torch.tensor([x1.grad, x2.grad, x3.grad])
assert torch.allclose(actual, expected, atol=1e-6)

To, co przed chwilą zrobiłeś ręcznie dla 4 węzłów wewnętrznych, PyTorch zrobił automatycznie — dla tego samego grafu, ale używając tej samej reguły. W realnych modelach graf ma miliony węzłów; algorytm jest ten sam.

### 1.7 Akumulacja gradientów

Wywołaj `backward()` po raz drugi (zbuduj graf jeszcze raz, bo PyTorch zwalnia go po pierwszym backward) i obejrzyj `x1.grad`, `x2.grad`, `x3.grad`. Co się stało?

In [ ]:
#| eval: false
a = x1 + x2
b = x2 * x3
c = torch.sin(b)
L = a * c
L.backward()

print(x1.grad, x2.grad, x3.grad)

Wartości się **podwoiły**. PyTorch nie zastępuje zawartości `.grad`, tylko ją dodaje — celowo, bo czasem chcemy zsumować gradienty z wielu wywołań backward przed aktualizacją. W typowej pętli treningowej musimy więc pamiętać, żeby wyzerować gradienty:

In [ ]:
#| eval: false
x1.grad.zero_()
x2.grad.zero_()
x3.grad.zero_()

W sekcji 3 zobaczysz, że `optim.SGD` (i każdy inny optymalizator) dostarcza metody `zero_grad()`, która robi to dla wszystkich parametrów modelu na raz.

### 1.8 `torch.no_grad()` i `.detach()`

Wykonaj jedną i tę samą operację — np. predykcję `y = a * c` — w trzech kontekstach i zaobserwuj `y.requires_grad` oraz `y.grad_fn`:

1. Zwykły kontekst: `y = a * c`.
2. W bloku `with torch.no_grad(): y = a * c`.
3. Po odpięciu od grafu: `y = (a * c).detach()`.

Pierwszy przypadek śledzi operację (graf się buduje), pozostałe dwa nie. `no_grad()` używamy, gdy w danym fragmencie kodu nie planujemy liczyć gradientu (walidacja, predykcja na nowych danych) — pozwala to PyTorchowi **nie zapamiętywać wartości pośrednich** potrzebnych do backward, oszczędzając pamięć. `.detach()` odpina pojedynczy tensor — przydaje się np. przy konwersji do NumPy do wizualizacji.

## 2. Regresja logistyczna z autograd — breast cancer

Wracamy do zbioru Breast Cancer Wisconsin z lab 8 (sekcja 3). Powtarzamy trening regresji logistycznej, ale tym razem gradientu nie liczymy ręcznie ze wzoru $\nabla_{\mathbf{w}} L = X^T(\sigma(X\mathbf{w}) - \mathbf{y})/N$ — odda nam tę pracę autograd.

### 2.1 Dane

Powtórz przygotowanie danych z lab 8 (sekcja 3.1). W skrócie:

1. `data = load_breast_cancer()` (z `sklearn.datasets`).
2. Podział train/test 80/20, `random_state=42`.
3. `StandardScaler` dopasowany na train, transformacja train i test.
4. Doklej kolumnę jedynek (bias) do macierzy cech.
5. Konwersja na tensory `float32`.
6. `TensorDataset` + `DataLoader` z `batch_size=32`, `shuffle=True` na zbiorze treningowym. Zbiór testowy zostaw jako surowe tensory.

### 2.2 Pętla treningowa z autograd

Wagi modelu to teraz tensor z `requires_grad=True`:

In [ ]:
#| eval: false
n_features_with_bias = X_train_t.shape[1]  # 30 + 1
w = (torch.randn(n_features_with_bias)* 0.01).requires_grad_(True)

*Uwaga:* mnożenie przez 0.01 daje lekko pomniejszoną inicjalizację. Niestety, nie można wykonać tej operacji bezpośrednio na tensorze z `requires_grad=True`, bo to odpina tensor od grafu. 

Funkcje pomocnicze (zgodnie z lab 8 sekcja 3.2):

- `sigmoid(z)` zaimplementowana jako `1 / (1 + torch.exp(-z))`,
- `bce_loss(y_true, y_pred)` z `torch.clamp` na $[\varepsilon, 1-\varepsilon]$ przed logarytmem.

Pętla treningowa z automatycznym różniczkowaniem:

In [ ]:
#| eval: false
lr = 0.1
num_epochs = 100
history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(num_epochs):
    for X_batch, y_batch in train_loader:
        # forward
        logits = X_batch @ w
        y_pred = sigmoid(logits)
        loss = bce_loss(y_batch, y_pred)

        # backward — tu autograd robi za nas to, co w lab 8
        # liczyliśmy ręcznie wzorem X^T (y_pred - y) / B
        loss.backward()

        # aktualizacja wag w torch.no_grad() — nie chcemy, żeby aktualizacja
        # trafiła do grafu obliczeniowego
        with torch.no_grad():
            w -= lr * w.grad
            w.grad.zero_()

    # ewaluacja po epoce — bez śledzenia grafu
    with torch.no_grad():
        # ... policz BCE i accuracy na train i test
        pass

Dwie nowe rzeczy w stosunku do lab 8:

- `loss.backward()` zastępuje ręczne wyprowadzenie i implementację gradientu — jedna linijka działa dla **dowolnej** funkcji straty, nie tylko BCE.
- `with torch.no_grad():` wokół aktualizacji wag jest konieczne. Bez tego operacja `w -= lr * w.grad` zostałaby wpisana do grafu, który PyTorch chce różniczkować przy następnym `backward()` — i rzuciłby błędem. Wewnątrz `no_grad` graf nie jest budowany, więc aktualizacja "in-place" jest bezpieczna.

### 2.3 Krzywe uczenia i porównanie z lab 8

Narysuj BCE train/test i accuracy train/test w funkcji epoki. Wypisz końcową accuracy na zbiorze testowym. Wynik powinien być **bardzo zbliżony** do tego z lab 8 sekcja 3.4 — to ten sam algorytm, ten sam zbiór, ta sama strata. Różnica polega tylko na tym, kto liczy gradient.

## 3. `nn.Module` i `optim` — kanoniczna pętla treningowa

W sekcji 2 nadal trzymaliśmy parametry modelu jako "luźny" tensor `w`, sami inicjalizowaliśmy go, sami zerowaliśmy gradient i sami pisaliśmy aktualizację. PyTorch oferuje dwa abstrakty, które porządkują ten kod: **`nn.Module`** opakowuje parametry modelu i forward pass, a **`optim`** opakowuje aktualizację wag.

### 3.1 Model jako klasa `nn.Module`

Zdefiniuj klasę `LogReg` dziedziczącą po `nn.Module`. W `__init__` utwórz pojedynczą warstwę `nn.Linear(n_features, 1)` (bias jest w `nn.Linear`, więc do macierzy danych nie doklejamy już kolumny jedynek). W `forward` zwróć **logity** (czyli `self.linear(x)`), bez sigmoid — funkcja straty zrobi sigmoid sama.

In [ ]:
#| eval: false
class LogReg(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x).squeeze(-1)  # kształt (B,) zamiast (B, 1)

Stwórz instancję modelu i wypisz `list(model.parameters())` — zobaczysz dwa tensory parametrów (waga i bias), oba z `requires_grad=True`, oba zainicjalizowane wariantem inicjalizacji Kaiminga.

### 3.2 Funkcja straty i optymalizator

Użyj `nn.BCEWithLogitsLoss` zamiast pisanej własnoręcznie BCE z sigmoidą. Jak omawialiśmy na wykładzie 5, łączy ona sigmoidę z log-em w sposób stabilny numerycznie — dlatego model zwraca logity, nie prawdopodobieństwa.

In [ ]:
#| eval: false
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

### 3.3 Kanoniczna pętla treningowa

In [ ]:
#| eval: false
num_epochs = 100
history = {"train_loss": [], "test_loss": [], "train_acc": [], "test_acc": []}

for epoch in range(num_epochs):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

    # ewaluacja
    model.eval()
    with torch.no_grad():
        # ... policz BCE i accuracy na train i test
        pass

Ten szkielet — `zero_grad → forward → loss → backward → step` — będziemy odtwarzać w każdym kolejnym treningu w tym kursie. Warto go zapamiętać.

Trzy nowe wywołania, które w sekcji 2 załatwialiśmy ręcznie:

- `optimizer.zero_grad()` zeruje gradienty wszystkich parametrów (`w.grad.zero_()` × liczba parametrów),
- `optimizer.step()` wykonuje aktualizację — dla `optim.SGD(lr=0.1)` to dokładnie `p -= 0.1 * p.grad` dla każdego parametru `p`, opakowane w `torch.no_grad()`,
- `model.train()` / `model.eval()` przełączają tryb modelu. Dla `nn.Linear` nie zmieniają nic, ale w sieciach z dropoutem (jeszcze poznamy) lub batch norm robią różnicę. Dobry zwyczaj — zawsze deklarować tryb explicite.

### 3.4 Trening i porównanie

Wytrenuj model i porównaj końcową accuracy z sekcją 2 — powinna być praktycznie taka sama. Ale **sekcja 3 jest krótsza w kodzie**: nie ma ręcznej inicjalizacji wag, nie ma własnej sigmoidy, nie ma własnej BCE, nie ma `with torch.no_grad()` wokół aktualizacji. Wszystko, co powtarzalne, schowaliśmy w abstrakcjach.

Od tej chwili będziemy używać `nn.Module` + `optim` w każdym treningu.

## 4. MLP na regresji nieliniowej — sunspoty

Wracamy do zbioru SILSO z lab 8 (sekcja 2), ale tym razem zamiast regresji liniowej z ręcznie skonstruowaną bazą Gaussa wytrenujemy **wielowarstwowy perceptron** (MLP), który nauczy się reprezentacji sam.

### 4.1 Dane

Powtórz wczytanie i przygotowanie danych z lab 8 (sekcja 2.1–2.2). W skrócie:

1. `pd.read_csv` z URL SILSO `https://www.sidc.be/SILSO/INFO/sndtotcsv.php`, parametry jak w lab 8.
2. Wyrzuć rekordy z brakującą wartością SN.
3. Skala czasu: $\tau = (t - t_{\min}) / (t_{\max} - t_{\min})$ z $t_{\min}, t_{\max}$ ustawionymi na ten sam zakres co w lab 8.
4. Podział na zbiór treningowy ($\tau < 0.8$) i walidacyjny ($\tau \geq 0.8$).
5. Konwersja na tensory `float32`. Cechy mają kształt `(N, 1)`, etykiety `(N,)`.
6. `TensorDataset` + `DataLoader` z `batch_size=256`, `shuffle=True`.

### 4.2 Model

Sieć z dwiema warstwami ukrytymi po 32 neurony, aktywacja ReLU. Najprościej — przez `nn.Sequential`:

In [ ]:
#| eval: false
model = nn.Sequential(
    nn.Linear(1, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 1),
)

`nn.Sequential` to skrót dla prostych modeli, w których forward pass to po prostu kompozycja warstw — nie trzeba pisać własnej klasy z `forward`. Wypisz `list(model.parameters())` — zobaczysz 6 tensorów (3 warstwy × waga i bias).

### 4.3 Strata, optymalizator i pętla treningowa

`nn.MSELoss` jako strata. Jako optymalizatora użyj `torch.optim.Adam(model.parameters(), lr=1e-3)`. Adam w odróżnieniu od SGD utrzymuje per-parametr średnie ruchome gradientu i jego kwadratu, dzięki czemu jest mniej wrażliwy na dobór `lr` niż SGD (na wykładzie 5 widzieliśmy go jako naturalny wybór "domyślny" dla sieci neuronowych).

Zastosuj kanoniczną pętlę treningową z sekcji 3.3. Liczba epok: 200. Pamiętaj o `model.eval()` + `torch.no_grad()` przy ewaluacji na zbiorze walidacyjnym.

*Uwaga o kształtach.* Wyjście `model(x)` ma kształt `(B, 1)`, a etykiety `(B,)`. `nn.MSELoss` nie zgłosi błędu, ale zacznie broadcastować do `(B, B)`, co da bezsensowny wynik. Spłaszcz wyjście: `model(x).squeeze(-1)`.

### 4.4 Wynik i porównanie z bazą Gaussa

1. Krzywa uczenia: train MSE i val MSE w funkcji epoki, oś $y$ w skali logarytmicznej.
2. Wykres dopasowania: na osi $x$ pełny zakres $\tau$, na osi $y$ predykcja modelu nałożona na dane (jak w lab 8 sekcja 2.7).

Porównaj z modelem z bazą Gaussa z lab 8 (sekcja 2.9.1 dla $K = 200$). Co możesz zaobserwować:

- W lab 8 musieliśmy ręcznie zaprojektować bazę: wybrać $K$, dobrać szerokość $s$, rozmieścić centra. MLP nie wymaga niczego z tej listy — sieć **uczy się reprezentacji** z surowego $\tau \in [0, 1]$.
- Dopasowanie powinno być wizualnie podobnej jakości lub lepsze.
- Liczba parametrów modelu: policz ją z `sum(p.numel() for p in model.parameters())` i porównaj z $K = 200$ z lab 8.

To pierwszy lab, w którym sieć "ma sens" — pokazuje swoją przewagę nad modelem ręcznie zaprojektowanym przez nas.


## 5. Klasyfikacja wieloklasowa — zbiór 2D z czterema klasami

W ostatniej sekcji zobaczymy klasyfikację wieloklasową w wykonaniu MLP. Użyjemy dwuwymiarowego, syntetycznego zbioru z czterema klasami, dla którego — dzięki tej dwuwymiarowości — możemy narysować **granice decyzyjne** modelu i naocznie porównać model liniowy z MLP.

### 5.1 Dane

Wczytaj zbiór z dwóch plików CSV. Każdy plik ma kolumny `x1`, `x2`, `y`, gdzie `y` przyjmuje wartości $0, 1, 2, 3$.

In [ ]:
#| eval: false
URL_TRAIN = "https://raw.githubusercontent.com/rodakt/BPwAD/refs/heads/v2/data/D2_class/train.csv"
URL_TEST  = "https://raw.githubusercontent.com/rodakt/BPwAD/refs/heads/v2/data/D2_class/test.csv"

df_train = pd.read_csv(URL_TRAIN)
df_test  = pd.read_csv(URL_TEST)

Narysuj scatter plot zbioru treningowego pokolorowany po klasach (np. `plt.scatter(..., c=y, cmap='tab10')`). Zobaczysz cztery struktury: pierścień, dwa skupiska o rozkładach Gaussa i półokrąg.

Zwróć uwagę, że pierścień (klasa 0) i półokrąg (klasa 3) **częściowo na siebie nachodzą** w dolnej połowie obrazka. Idealne accuracy nie jest tu osiągalne — to nie wada modelu, tylko właściwość zbioru. Zobaczymy ją w macierzy pomyłek w 5.7.

### 5.2 Przygotowanie tensorów

1. `StandardScaler` dopasowany na train, transformacja train i test.
2. Cechy jako `float32`, etykiety jako **`torch.long`** — `nn.CrossEntropyLoss` wymaga indeksów klas typu całkowitego.
3. `TensorDataset` + `DataLoader` z `batch_size=64`, `shuffle=True` na zbiorze treningowym. Zbiór testowy zostaw jako surowe tensory.

### 5.3 Baseline: regresja softmax

Zacznij od najprostszego możliwego modelu — pojedynczej warstwy liniowej z 4 wyjściami:

In [ ]:
#| eval: false
model_lin = nn.Linear(2, 4)

To **regresja softmax** (wieloklasowy odpowiednik regresji logistycznej): logity przechodzą przez softmax, dając rozkład prawdopodobieństwa nad 4 klasami. Tak jak `nn.BCEWithLogitsLoss` w sekcji 3, `nn.CrossEntropyLoss` łączy softmax z entropią krzyżową w jednej operacji stabilnej numerycznie — model zwraca logity, nie prawdopodobieństwa.

In [ ]:
#| eval: false
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_lin.parameters(), lr=1e-2)

Wytrenuj model przez 100 epok w kanonicznej pętli (sekcja 3.3). Wypisz końcową accuracy na zbiorze testowym (predykcja klasy: `logits.argmax(dim=1)`).

### 5.4 Wizualizacja granic decyzyjnych

Napisz funkcję `plot_decision_boundary(model, X, y, ax)`, która:

1. Tworzy gęstą siatkę punktów na obszarze obejmującym cały zbiór (np. `np.meshgrid` na zakresie `[X.min - 0.5, X.max + 0.5]` z krokiem 0.02).
2. Konwertuje siatkę na tensor `float32`.
3. W trybie `model.eval()` + `torch.no_grad()` liczy `logits = model(grid)` i `Z = logits.argmax(dim=1)`.
4. Rysuje `ax.contourf(xx, yy, Z, alpha=0.3, cmap='tab10', levels=...)` — kolorowane regiony to obszary, gdzie model przewiduje daną klasę.
5. Nakłada `ax.scatter` z punktami zbioru.

*Wskazówka dot. siatki.* Pamiętaj, że model był trenowany na danych po standaryzacji — siatkę i punkty rysujemy w **przestrzeni po standaryzacji** (tak, by model widział je tak samo jak dane treningowe). Albo rysujesz w oryginalnej skali, ale wtedy przed predykcją musisz przepuścić siatkę przez ten sam `StandardScaler`.

Narysuj granice dla `model_lin`. Co widzisz?

Granice modelu liniowego to kawałki płaszczyzn — w 2D to **linie proste**. Cztery klasy → cztery regiony oddzielone liniami. Pierścień (klasa 0) jest niemożliwy do oddzielenia liniami od klas wewnątrz/na zewnątrz — model musi wybrać jakieś "kompromisowe" cięcia, które nie odzwierciedlają struktury danych.

### 5.5 MLP

Wprowadź dwie warstwy ukryte:

In [ ]:
#| eval: false
model_mlp = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Linear(32, 32),
    nn.ReLU(),
    nn.Linear(32, 4),
)

Wytrenuj go w **identycznej** pętli co w 5.3 (z osobnym optymalizatorem dla nowego modelu). Wypisz końcową accuracy.

### 5.6 Porównanie side-by-side

Narysuj dwa wykresy obok siebie (`fig, (ax1, ax2) = plt.subplots(1, 2)`):

- po lewej granice `model_lin`,
- po prawej granice `model_mlp`.

Granice MLP powinny być **zakrzywione** — pierścień zostaje znaleziony jako oddzielny region, a granice między klasami dopasowują się do faktycznej struktury danych. To naoczna ilustracja tego, co warstwa ukryta z nieliniowością wnosi do modelu: zdolność wyrażania **funkcji nieliniowych**, których model liniowy nie umie wyrazić w żaden sposób.

### 5.7 Macierz pomyłek

Dla obu modeli policz macierz pomyłek na zbiorze testowym (`sklearn.metrics.confusion_matrix`). Porównaj:

- W modelu liniowym błędy są rozsiane po wielu klasach.
- W MLP błędy są **skoncentrowane na granicy klas 0 i 3** (pierścień i półokrąg) — tam, gdzie zbiór faktycznie zawiera próbki nie do rozróżnienia. Reszta klas powinna być rozdzielana niemal bezbłędnie.

To dobra okazja, żeby zauważyć: jakość modelu nie sprowadza się do jednej liczby. Macierz pomyłek pokazuje, **gdzie** model się myli i czy te miejsca mają sens.


## 6. Zadania dodatkowe

### 6.1 Wpływ szerokości warstwy ukrytej na granice decyzyjne

Wytrenuj kilka MLP-ów z jedną warstwą ukrytą o różnej szerokości — `hidden_size ∈ {2, 4, 8, 32, 128}`. Dla każdego narysuj granice decyzyjne na zbiorze 2D z sekcji 5. Ułóż wykresy w siatce subplotów (np. 1×5 lub 2×3).

Co obserwujesz, gdy szerokość rośnie? W którym momencie pierścień zostaje "znaleziony"? Czy przy `hidden_size = 128` granice nie stają się nadmiernie pofalowane (przeuczenie)?

### 6.2 Adam vs SGD na sunspot-ach

Wytrenuj sieć z sekcji 4 dwoma optymalizatorami: `optim.SGD(lr=0.01)` i `optim.Adam(lr=1e-3)`. Pozostałe hiperparametry takie same. Narysuj krzywe zbieżności (val MSE w funkcji epoki) na jednym wykresie.

Spróbuj też `optim.SGD` z różnymi `lr` ($10^{-1}, 10^{-2}, 10^{-3}$). Czy któreś `lr` daje SGD wynik porównywalny z Adamem? Czy strojenie `lr` jest pracochłonne?

### 6.3 Regresja logistyczna ze scikit-learn — kontrola

Wytrenuj `sklearn.linear_model.LogisticRegression` na danych z sekcji 2/3 i porównaj accuracy z modelem z sekcji 3. Dla regresji logistycznej na małym zbiorze sklearn używa metody L-BFGS, więc nie strojąc niczego dostaniesz wynik referencyjny. Twój PyTorch-owy model po 100 epokach SGD powinien osiągnąć podobną accuracy — różnica rzędu 1 punktu procentowego jest normalna.